In [1]:
import requests
import pandas as pd
import json
import os
import boto3
import scrapy
from dotenv import load_dotenv


csv_url="https://tmopenlabbucket.s3.eu-west-3.amazonaws.com/City_Meteo_Rank.csv"


df = pd.read_csv(csv_url,index_col=0)

In [2]:
list_cities=df['city'].to_list()

In [ ]:
import subprocess


# Liste des villes à envoyer à script.py
cities = list_cities

# Construire l'argument en ligne de commande
cmd = ['python3','booking_scrap_final.py','--cities'] + cities

# Exécuter le script en passant les villes comme argument
result = subprocess.run(cmd)


In [4]:
df_booking=pd.read_json('hotels.json')
df_city_ccm=pd.read_csv('cities_lat_long_ccm.csv',index_col=0)


In [5]:
df_join=df_city_ccm.merge(df_booking, left_on='city', right_on='city', how='left')
df_join=df_join.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

In [6]:
df_join

,city,lat,lon,CCM,name,url,score,description,latitude,longitude
0,Le Havre,49.493898,0.107973,0.925,"The Originals Boutique, Hôtel d'Angleterre, Le...",https://www.booking.com/hotel/fr/comfort-d-ang...,7.6,This hotel is located in the town centre of Le...,49.494049,0.099366
1,Le Havre,49.493898,0.107973,0.925,LA PARENTHÈSE HAVRAISE - Parking privé Plein c...,https://www.booking.com/hotel/fr/la-parenthese...,9.1,LA PARENTHÈSE HAVRAISE - Parking privé Plein c...,49.496836,0.106893
2,Le Havre,49.493898,0.107973,0.925,Best Western ARThotel,https://www.booking.com/hotel/fr/art.en-gb.htm...,8.0,The Best Western Art Hotel is located in the h...,49.491194,0.106461
3,Le Havre,49.493898,0.107973,0.925,Aparthotel Adagio Access Le Havre Les Docks,https://www.booking.com/hotel/fr/adagio-access...,8.7,"Located in Le Havre, Aparthotel Adagio Access ...",49.487419,0.131511
4,Le Havre,49.493898,0.107973,0.925,Best Western Plus Le Havre Centre Gare,https://www.booking.com/hotel/fr/hotelterminus...,8.3,None,49.493344,0.124318
...,...,...,...,...,...,...,...,...,...,...
305,Avignon,43.949249,4.805901,0.550,Mercure Avignon Gare TGV,https://www.booking.com/hotel/fr/expresshiavig...,8.5,None,43.929290,4.784118
306,Avignon,43.949249,4.805901,0.550,Hotel De Cambis Best Western Premier Collection,https://www.booking.com/hotel/fr/de-cambis-bw-...,8.7,"Well set in Avignon, Hotel De Cambis Best West...",43.947113,4.803330
307,Avignon,43.949249,4.805901,0.550,Hôtel Cloitre Saint Louis Avignon,https://www.booking.com/hotel/fr/clarioncloitr...,7.6,"Set in central Avignon, a 15-minute walk from ...",43.943808,4.804947
308,Avignon,43.949249,4.805901,0.550,Best Western Plus Le Lavarin,https://www.booking.com/hotel/fr/hotellelavari...,8.3,None,43.922758,4.804295


In [7]:
df_join.to_csv('City_Meteo_Rank_Booking.csv')

In [8]:
### Upload to S3

load_dotenv()

aws_access_key_id =os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key =os.getenv('AWS_SECRET_ACCESS_KEY')
s3 = boto3.resource('s3',aws_access_key_id=aws_access_key_id,aws_secret_access_key=aws_secret_access_key)
bucket=s3.Bucket('tmopenlabbucket')

In [9]:
bucket.upload_file(Filename='City_Meteo_Rank_Booking.csv',Key='City_Meteo_Rank_Booking.csv', ExtraArgs={'ACL':'public-read'})